# Part → Assembly Workflow

This notebook demonstrates the **Abaqus-style Part/Assembly** workflow
in pyGmsh:

1. **Build Parts** in isolated Gmsh sessions → export to STEP
2. **Assemble** — import Parts, position, fragment for conformal interfaces
3. **Mesh** — transfinite quad mesh, physical groups
4. **Number** — contiguous solver-ready IDs via the Numberer (with optional RCM bandwidth optimisation)
5. **Constrain** — define and resolve multi-point constraints
6. **Analyse** — eigenvalue analysis in OpenSees

## Example: IPE200 from three plate Parts

```
        ┌─────── b=100 ───────┐
        │  tf=8.5  (top fl.)  │   ← flange Part, translated to z = d
        └────────┬────────────┘
                 │
            tw=5.6│  hw = d = h - tf
                 │        = 191.5
                 │
        ┌────────┴────────────┐
        │  tf=8.5  (bot fl.)  │   ← same flange Part at z = 0
        └─────────────────────┘
```

Units: **mm, N, s, tonne**

In [ ]:
from pyGmsh import Part, Assembly, Numberer, NumberedMesh, Constraints
import numpy as np

## 1 — Define IPE200 parameters

In [ ]:
# IPE200 cross-section (all in mm)
h   = 200.0     # total depth
b   = 100.0     # flange width
tw  = 5.6       # web thickness
tf  = 8.5       # flange thickness
L   = 3000.0    # beam length (3 m)

# Derived: distance between flange mid-surfaces
d = h - tf      # = 191.5 mm

print(f"IPE200: h={h}, b={b}, tw={tw}, tf={tf} mm")
print(f"Beam length: {L} mm")
print(f"Flange mid-surface spacing: d = {d} mm")

## 2 — Build the Web Part

A flat rectangular plate in the XZ plane: length L along X, height d
along Z, centred on Y = 0.  This is the *mid-surface* of the web.

In [ ]:
# ── Web Part ─────────────────────────────────────────────────
web = Part("web")
web.begin(verbose=True)

# Rectangle in XZ plane: x ∈ [0, L], y = 0, z ∈ [0, d]
p1 = web.model.add_point(0, 0, 0,   sync=False)
p2 = web.model.add_point(L, 0, 0,   sync=False)
p3 = web.model.add_point(L, 0, d,   sync=False)
p4 = web.model.add_point(0, 0, d,   sync=False)

l1 = web.model.add_line(p1, p2, sync=False)
l2 = web.model.add_line(p2, p3, sync=False)
l3 = web.model.add_line(p3, p4, sync=False)
l4 = web.model.add_line(p4, p1, sync=False)

loop = web.model.add_curve_loop([l1, l2, l3, l4], sync=False)
surf = web.model.add_plane_surface(loop)      # sync=True here

# Store thickness as Part metadata
web.properties['thickness'] = tw
web.properties['material']  = 'steel'

print(f"Web surface tag: {surf}")
print(f"Web properties: {web.properties}")

In [ ]:
# Plot the web part geometry
web.plot.geometry(label_tags=True, show=True)

In [ ]:
# Save the web to STEP and close the session
web_file = web.save("web.step")
web.end()

print(f"Web saved to: {web_file}")
print(f"Part status: {web}")

## 3 — Build the Flange Part

A flat rectangular plate in the XY plane: length L along X, width b
along Y (centred at Y = 0).  This plate will be *reused* for both
the bottom and top flanges — just translated to different Z positions.

In [ ]:
# ── Flange Part ──────────────────────────────────────────────
flange = Part("flange")
flange.begin(verbose=True)

# Rectangle in XY plane: x ∈ [0, L], y ∈ [-b/2, +b/2], z = 0
p1 = flange.model.add_point(0,  -b/2, 0, sync=False)
p2 = flange.model.add_point(L,  -b/2, 0, sync=False)
p3 = flange.model.add_point(L,  +b/2, 0, sync=False)
p4 = flange.model.add_point(0,  +b/2, 0, sync=False)

l1 = flange.model.add_line(p1, p2, sync=False)
l2 = flange.model.add_line(p2, p3, sync=False)
l3 = flange.model.add_line(p3, p4, sync=False)
l4 = flange.model.add_line(p4, p1, sync=False)

loop = flange.model.add_curve_loop([l1, l2, l3, l4], sync=False)
surf = flange.model.add_plane_surface(loop)

flange.properties['thickness'] = tf
flange.properties['material']  = 'steel'

print(f"Flange surface tag: {surf}")
print(f"Flange properties: {flange.properties}")

In [ ]:
# Plot the flange part geometry
flange.plot.geometry(label_tags=True, show=True)

In [ ]:
# Save and close
flange_file = flange.save("flange.step")
flange.end()

print(f"Flange saved to: {flange_file}")
print(f"Part status: {flange}")

## 4 — Create the Assembly

Open a fresh Gmsh session and import both Parts:

| Instance       | Part   | Translation      | Role             |
|:---------------|:-------|:-----------------|:-----------------|
| `web`          | web    | (0, 0, 0)        | web plate        |
| `bot_flange`   | flange | (0, 0, 0)        | bottom flange    |
| `top_flange`   | flange | (0, 0, d)        | top flange       |

The web sits in the XZ plane (Y = 0).  Flanges lie in XY planes at
Z = 0 and Z = d.  Their mid-lines overlap the web at Y = 0, creating
shared edges that will become conformal after fragmentation.

In [ ]:
# ── Assembly ─────────────────────────────────────────────────
asm = Assembly("IPE200_assembly")
asm.begin(verbose=True)

# Import web (identity placement)
i_web = asm.add_part(web, label="web")
print(f"Web instance:  {i_web}")

# Import bottom flange (same z=0 as built)
i_bot = asm.add_part(flange, label="bot_flange")
print(f"Bot flange:    {i_bot}")

# Import top flange (shift to z=d)
i_top = asm.add_part(flange, label="top_flange",
                     translate=(0, 0, d))
print(f"Top flange:    {i_top}")

print(f"\nAssembly: {asm}")
print(f"Instances: {asm.list_instances()}")

## 5 — Fragment for conformal mesh

The three surfaces overlap along two horizontal edges (the web-flange
junctions at z = 0 and z = d).  `fragment_all()` splits all surfaces
at their intersections so the mesh will be conformal (shared nodes at
the junction lines).

After fragmentation, the original 3 surfaces become 5 sub-surfaces:

```
  ┌─────┬─────┐  top flange: 2 halves (split by web)
        │
        │         web: full height
        │
  ┌─────┴─────┐  bottom flange: 2 halves
```

In [ ]:
# Fragment all surfaces for conformal mesh
surviving = asm.fragment_all(dim=2)
print(f"After fragmentation: {len(surviving)} surfaces")
print(f"Surface tags: {surviving}")

# Verify: should be 5 surfaces (2 bottom-flange halves,
#         1 web, 2 top-flange halves)

In [ ]:
# Plot the assembled geometry after fragmentation
asm.plot.geometry(label_tags=True, show=True)

## 6 — Mesh the Assembly

We use the pyGmsh mesh API on the assembly's context to set up
transfinite meshing for a structured quad mesh, then generate.

In [ ]:
import gmsh

# ── Mesh control ─────────────────────────────────────────────
# Target mesh divisions
n_length  = 60   # along beam axis (X)
n_web     = 12   # along web height (Z)
n_flange  = 4    # across each flange half (Y)

# Classify curves by their bounding box
for dim_c, tag_c in gmsh.model.getEntities(1):
    bb = gmsh.model.getBoundingBox(dim_c, tag_c)
    xmin, ymin, zmin, xmax, ymax, zmax = bb
    dx = xmax - xmin
    dy = ymax - ymin
    dz = zmax - zmin

    length = max(dx, dy, dz)
    tol = 1.0  # 1 mm tolerance

    if abs(length - L) < tol:           # long edges along X
        gmsh.model.mesh.setTransfiniteCurve(tag_c, n_length + 1)
    elif abs(length - d) < tol:         # web height edges
        gmsh.model.mesh.setTransfiniteCurve(tag_c, n_web + 1)
    elif abs(length - b/2) < tol:       # flange half-width edges
        gmsh.model.mesh.setTransfiniteCurve(tag_c, n_flange + 1)

# Transfinite surfaces + recombine for all-quad mesh
for dim_s, tag_s in gmsh.model.getEntities(2):
    gmsh.model.mesh.setTransfiniteSurface(tag_s)
    gmsh.model.mesh.setRecombine(2, tag_s)

# Generate 2D mesh
gmsh.model.mesh.generate(2)

# Report
node_tags_raw, _, _ = gmsh.model.mesh.getNodes()
print(f"Mesh generated: {len(node_tags_raw)} nodes")

In [ ]:
# Plot the assembly mesh
asm.plot.mesh(show=True)

## 7 — Assign physical groups

Physical groups are created directly from instance labels — 
`fragment_all()` updates each instance's entity tags automatically,
so no bounding-box heuristics are needed.

In [ ]:
# Physical groups directly from instance labels (entities updated by fragment_all)
groups = asm.add_physical_groups_from_instances(dim=2)

# Keep references for the OpenSees section assignment later
web_surfs    = asm.instances["web"].entities[2]
flange_surfs = (asm.instances["bot_flange"].entities[2]
              + asm.instances["top_flange"].entities[2])

print(f"Physical groups: {groups}")
print(f"Web surfaces:    {web_surfs}")
print(f"Flange surfaces: {flange_surfs}")

## 8 — Extract FEM data and Renumber

Two options:

1. **`get_fem_data()`** — raw Gmsh tags (non-contiguous)
2. **`get_numbered_mesh()`** — contiguous solver-ready IDs via the Numberer

We use option 2 for the solver.  The `NumberedMesh` object carries
bidirectional maps (`gmsh_to_solver_node`, `solver_to_gmsh_node`, etc.)
so post-processing can translate results back to Gmsh tags.

### Numbering methods

| Method     | Description                                        |
|:-----------|:---------------------------------------------------|
| `"simple"` | Contiguous 1-based IDs, preserving relative order  |
| `"rcm"`    | Reverse Cuthill-McKee bandwidth optimisation        |

In [ ]:
# ── Option A: raw FEM data (Gmsh tags) ─────────────────────────
fem_raw = asm.mesh.get_fem_data(dim=2)
print("Raw FEM data (Gmsh tags):")
print(f"  node_tags shape:    {fem_raw['node_tags'].shape}")
print(f"  connectivity shape: {fem_raw['connectivity'].shape}")
print(f"  elem_tags count:    {len(fem_raw['elem_tags'])}")
print(f"  used nodes:         {len(fem_raw['used_tags'])}")
print()

In [ ]:
# ── Option B: numbered mesh (solver-ready) ─────────────────────
# Compare methods first
numb = Numberer(fem_raw)
bw_comparison = numb.compare_methods()
print("Bandwidth comparison:")
for method, bw in bw_comparison.items():
    print(f"  {method:>6s}: bandwidth = {bw}")
print()

# Use the convenience method (picks best)
mesh = asm.mesh.get_numbered_mesh(dim=2, method="simple")
print(mesh.summary())
print(f"\nNode ID range:    [{mesh.node_ids.min()}, {mesh.node_ids.max()}]")
print(f"Element ID range: [{mesh.element_ids.min()}, {mesh.element_ids.max()}]")
print(f"Coord ranges:")
print(f"  X: [{mesh.node_coords[:,0].min():.1f}, {mesh.node_coords[:,0].max():.1f}]")
print(f"  Y: [{mesh.node_coords[:,1].min():.1f}, {mesh.node_coords[:,1].max():.1f}]")
print(f"  Z: [{mesh.node_coords[:,2].min():.1f}, {mesh.node_coords[:,2].max():.1f}]")

In [ ]:
# ── Bidirectional maps ───────────────────────────────
# Translate between Gmsh tags and solver IDs
print("Map examples (first 5 entries):")
for i, (gtag, sid) in enumerate(list(mesh.gmsh_to_solver_node.items())[:5]):
    print(f"  Gmsh node {gtag} → solver node {sid}")

print()
for i, (sid, gtag) in enumerate(list(mesh.solver_to_gmsh_node.items())[:5]):
    print(f"  Solver node {sid} → Gmsh node {gtag}")

## 9 — Build OpenSees model

The numbered mesh maps directly to ShellMITC4 elements.  We create
two sections (web thickness and flange thickness) and assign them
based on which physical group each element belongs to.

Note: we use the `gmsh_to_solver_elem` map to figure out which
solver element IDs correspond to web vs flange physical groups.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

# ── Material ─────────────────────────────────────────────────
E   = 210000.0    # MPa (N/mm²)
nu  = 0.3
rho = 7.85e-9     # tonne/mm³

# Two sections: web and flange, different thicknesses
sec_web    = 1
sec_flange = 2
ops.section('ElasticMembranePlateSection', sec_web,    E, nu, tw, rho)
ops.section('ElasticMembranePlateSection', sec_flange, E, nu, tf, rho)

# ── Nodes (from NumberedMesh) ─────────────────────────────
for i in range(mesh.n_nodes):
    nid = int(mesh.node_ids[i])
    x, y, z = mesh.node_coords[i]
    ops.node(nid, float(x), float(y), float(z))

print(f"Created {mesh.n_nodes} nodes")

# ── Build web/flange element sets via the maps ───────────
# Query Gmsh for element tags per physical group surface
web_solver_eids    = set()
flange_solver_eids = set()

for surf_tag in web_surfs:
    etypes, etags, _ = gmsh.model.mesh.getElements(2, surf_tag)
    for et in etags:
        for gmsh_eid in et:
            solver_eid = mesh.gmsh_to_solver_elem.get(int(gmsh_eid))
            if solver_eid is not None:
                web_solver_eids.add(solver_eid)

for surf_tag in flange_surfs:
    etypes, etags, _ = gmsh.model.mesh.getElements(2, surf_tag)
    for et in etags:
        for gmsh_eid in et:
            solver_eid = mesh.gmsh_to_solver_elem.get(int(gmsh_eid))
            if solver_eid is not None:
                flange_solver_eids.add(solver_eid)

# ── Elements (from NumberedMesh) ──────────────────────────
for i in range(mesh.n_elems):
    eid   = int(mesh.element_ids[i])
    nodes = [int(n) for n in mesh.connectivity[i]]
    sec   = sec_web if eid in web_solver_eids else sec_flange
    ops.element('ShellMITC4', eid, *nodes, sec)

print(f"Created {mesh.n_elems} ShellMITC4 elements")
print(f"  Web elements:    {len(web_solver_eids)}")
print(f"  Flange elements: {len(flange_solver_eids)}")

## 10 — Cantilever eigenvalue analysis

Quick sanity check: fix one end, compute the first few natural
frequencies.  We use `solver_to_gmsh_node` to identify boundary
nodes by their coordinates.

In [ ]:
# ── Boundary conditions: fix x = 0 ───────────────────────
tol = 0.1
n_fixed = 0
for i in range(mesh.n_nodes):
    x = mesh.node_coords[i, 0]
    if x < tol:
        ops.fix(int(mesh.node_ids[i]), 1, 1, 1, 1, 1, 1)
        n_fixed += 1

print(f"Fixed {n_fixed} nodes at x = 0")

# ── Eigenvalue analysis ─────────────────────────────────
n_modes = 6
eigenvalues = ops.eigen(n_modes)

print(f"\n{'Mode':>4}  {'ω² (rad²/s²)':>16}  {'f (Hz)':>10}")
print("-" * 36)
for i, ev in enumerate(eigenvalues):
    omega = np.sqrt(abs(ev))
    freq  = omega / (2 * np.pi)
    print(f"{i+1:>4}  {ev:>16.4f}  {freq:>10.3f}")

## 11 — Visualise mode shapes in Gmsh

Extract mode shapes from OpenSees and display them as Gmsh vector views.
Note: we translate solver node IDs back to Gmsh tags using the
`solver_to_gmsh_node` map so Gmsh can locate the vectors.

In [ ]:
# Extract mode shapes and show mode 1 as a Gmsh view
gmsh_node_tags = np.array(
    [mesh.solver_to_gmsh_node[int(sid)] for sid in mesh.node_ids],
    dtype=int,
)

mode_shapes = []
for m in range(n_modes):
    disp = np.zeros((mesh.n_nodes, 3))
    for i in range(mesh.n_nodes):
        solver_nid = int(mesh.node_ids[i])
        ev = ops.nodeEigenvector(solver_nid, m + 1)
        disp[i] = ev[:3]  # ux, uy, uz
    mode_shapes.append(disp)

# Show first mode as vector view (using Gmsh tags)
asm.view.add_node_vector(
    name="Mode_1",
    node_tags=gmsh_node_tags,
    vectors=mode_shapes[0],
)

print("Mode 1 view added to Gmsh")
print(f"Max displacement: {np.abs(mode_shapes[0]).max():.6f}")

## 12 — Constraints API

The Assembly provides a full solver-agnostic constraint engine with
a **two-stage pipeline**:

1. **Stage 1 — Definition** (pre-mesh): declare constraint *intent*
2. **Stage 2 — Resolution** (post-mesh): convert to concrete node pairs,
   weights, and constraint matrices

### Constraint taxonomy

| Level | Type                  | Description                                   |
|:------|:----------------------|:----------------------------------------------|
| 1     | `equal_dof`           | Co-located nodes share selected DOFs           |
| 1     | `rigid_link`          | Rigid bar (beam or rod)                        |
| 1     | `penalty`             | Soft spring between co-located nodes           |
| 2     | `rigid_diaphragm`     | In-plane DOFs follow master                    |
| 2     | `rigid_body`          | All 6 DOFs follow master rigidly               |
| 2     | `kinematic_coupling`  | User picks which DOFs are constrained          |
| 3     | `tie`                 | Shape function interpolation on master face    |
| 3     | `distributing_coupling` | Load at master distributed to slave surface |
| 3     | `embedded`            | Embedded element nodes follow host field       |
| 4     | `tied_contact`        | Full surface-to-surface tie                    |
| 4     | `mortar`              | Lagrange multiplier coupling                   |

All constraints ultimately express: **u_slave = C · u_master**

In [ ]:
# This I-beam doesn't need constraints (fragment makes interfaces
# conformal), but here's how the API works:

# ── Level 1: Node-to-Node ─────────────────────────────

# Equal DOF: slave nodes copy master DOFs
# asm.equal_dof(
#     master_label="web",
#     slave_label="bot_flange",
#     dofs=[1, 2, 3],       # translations only
#     tolerance=1.0,        # mm — how close nodes must be
# )

# Rigid link: rigid bar between master and slave
# asm.rigid_link(
#     master_label="web",
#     slave_label="bot_flange",
#     link_type="beam",     # "beam" = 6-DOF, "rod" = translations only
# )

# Penalty: soft spring between co-located nodes
# asm.penalty(
#     master_label="web",
#     slave_label="bot_flange",
#     stiffness=1e10,
#     dofs=[1, 2, 3],
# )

print("Level 1 (Node-to-Node): equal_dof, rigid_link, penalty")

In [ ]:
# ── Level 2: Node-to-Group ────────────────────────────

# Rigid diaphragm: floor slab constraint
# asm.rigid_diaphragm(
#     master_label="web",
#     slave_label="top_flange",
#     master_point=(L/2, 0, d),
#     plane_normal=(0, 0, 1),    # XY plane
#     constrained_dofs=[1, 2, 6], # ux, uy, rz
# )

# Rigid body: all DOFs follow master
# asm.rigid_body(
#     master_label="web",
#     slave_label="top_flange",
#     master_point=(L/2, 0, d),
# )

# Kinematic coupling: pick which DOFs
# asm.kinematic_coupling(
#     master_label="web",
#     slave_label="top_flange",
#     master_point=(L/2, 0, d),
#     dofs=[1, 2, 3],            # translations only
# )

print("Level 2 (Node-to-Group): rigid_diaphragm, rigid_body, kinematic_coupling")

In [ ]:
# ── Level 3: Node-to-Surface ──────────────────────────

# Tie: surface tie via shape function interpolation
# asm.tie(
#     master_label="web",
#     slave_label="bot_flange",
#     tolerance=1.0,
#     dofs=[1, 2, 3, 4, 5, 6],  # all DOFs
# )

# Distributing coupling: load at master distributed to slave surface
# asm.distributing_coupling(
#     master_label="web",
#     slave_label="bot_flange",
#     master_point=(L/2, 0, 0),
#     weighting="uniform",       # or "area"
# )

# Embedded: embedded element nodes follow host field
# asm.embedded(
#     host_label="web",
#     embedded_label="bot_flange",
#     tolerance=1.0,
# )

print("Level 3 (Node-to-Surface): tie, distributing_coupling, embedded")

In [ ]:
# ── Level 4: Surface-to-Surface ────────────────────────

# Tied contact: full surface-to-surface tie
# asm.tied_contact(
#     master_label="web",
#     slave_label="bot_flange",
#     tolerance=1.0,
# )

# Mortar: Lagrange multiplier coupling
# asm.mortar(
#     master_label="web",
#     slave_label="bot_flange",
#     integration_order=2,
# )

print("Level 4 (Surface-to-Surface): tied_contact, mortar")

In [ ]:
# ── Resolving constraints ─────────────────────────────
#
# After meshing + defining constraints, call:
#
#   records = asm.resolve_constraints(
#       node_tags=fem_raw['node_tags'],
#       node_coords=fem_raw['node_coords'],
#       elem_tags=fem_raw['elem_tags'],
#       connectivity=fem_raw['connectivity'],
#   )
#
# This converts definitions → concrete records:
#   - NodePairRecord      (Level 1: equal_dof, rigid_link, penalty)
#   - NodeGroupRecord     (Level 2: diaphragm, rigid_body, coupling)
#   - InterpolationRecord (Level 3: tie, distributing, embedded)
#   - SurfaceCouplingRecord (Level 4: tied_contact, mortar)
#
# Each record carries the constraint matrix C and weights needed
# by any solver (OpenSees, Abaqus, Code_Aster, etc.)

print("Constraint definitions:", asm.list_constraint_defs())
print("Resolved records:      ", asm.list_constraint_records())
print("(Both empty — this I-beam uses fragmentation, not constraints)")

## 13 — External file import

If you already have CAD files from SolidWorks, FreeCAD, etc., use
`add_file()` instead of `add_part()`.  Supports STEP, IGES, and
also `.msh` files (for pre-meshed components).

In [ ]:
# ── Example: import external files ───────────────────────
#
# asm2 = Assembly("external")
# asm2.begin()
#
# # CAD files (STEP/IGES): preserve parametric geometry
# i1 = asm2.add_file("column.step",
#                     label="column",
#                     translate=(0, 0, 0))
#
# i2 = asm2.add_file("beam.iges",
#                     label="beam",
#                     translate=(0, 0, 3000),
#                     rotate=(np.pi/2, 0, 1, 0))  # 90° about Y
#
# # Pre-meshed file (.msh): preserves mesh + physical groups
# # NOTE: .msh does NOT preserve parametric geometry (cannot re-mesh)
# i3 = asm2.add_file("slab.msh", label="slab")
#
# asm2.fragment_all()
# asm2.mesh.generate(dim=3)
#
# # Use the Numberer for solver-ready IDs
# mesh2 = asm2.mesh.get_numbered_mesh(dim=3, method="rcm")
# print(mesh2.summary())
#
# asm2.end()

print("External file import via add_file() — see examples above")

## 14 — Export assembled geometry

In [ ]:
# Export the assembled geometry
asm.model.save_step("IPE200_assembly.step")
asm.model.save_iges("IPE200_assembly.iges")

print("Assembly exported to STEP and IGES")

## 15 — Finalize

In [ ]:
# Launch GUI to inspect results
asm.model.gui()

# Clean up
asm.end()
print("Assembly session closed.")
print(f"Final status: {asm}")

---

## Summary

| Concept          | Abaqus              | pyGmsh                          |
|:-----------------|:--------------------|:--------------------------------|
| Geometry unit    | `Part`              | `Part` (own Gmsh session)       |
| CAD export       | —                   | `.save("file.step")`            |
| Assembly         | `Assembly` module   | `Assembly` class                |
| Instance         | `Instance`          | `add_part()` → `Instance`       |
| Placement        | `translate/rotate`  | `translate=`, `rotate=` kwargs  |
| Merge/tie        | `*TIE`, `*COUPLING` | `fragment_all()`, `tie()`, etc  |
| Mesh             | `Mesh` module       | `asm.mesh.generate()`           |
| Numbering        | `*NUMBERING`        | `asm.mesh.get_numbered_mesh()`  |
| Bandwidth opt    | —                   | `method="rcm"` (Reverse C-M)    |
| Physical groups  | `*ELSET`, `*NSET`   | `asm.physical.add()`            |
| Constraints      | `*MPC`, `*TIE`      | `asm.equal_dof()`, `asm.tie()`  |
| Constraint solve | internal            | `asm.resolve_constraints()`     |
| Post-processing  | ODB                 | `asm.view`, `solver_to_gmsh_node` |
| External CAD     | `Import Part`       | `asm.add_file("part.step")`     |
| GUI              | CAE                 | `asm.model.gui()`               |

### Key design decisions

- **Part = geometry only**: no mesh, no physical groups. Part creates only Model and Inspect composites. STEP is the default export (preserves parametric OCC geometry for re-meshing).
- **Assembly = everything else**: the Assembly creates all composites directly (Mesh, PhysicalGroups, View, Plot, Inspect, Model, Partition, Gmsh2OpenSees) — no intermediate wrapper. Access them as `asm.mesh`, `asm.physical`, `asm.view`, etc.
- **STEP is the contract**: Part and Assembly are fully decoupled. The only interface is the STEP file on disk.
- **Numberer**: remaps non-contiguous Gmsh tags to contiguous solver-ready IDs. Optional RCM bandwidth optimisation for direct solvers.
- **Bidirectional maps**: Gmsh tag ↔ solver ID for seamless post-processing (e.g. visualise OpenSees results back in Gmsh).
- **Constraints are solver-agnostic**: two-stage pipeline (definition → resolution) produces records that any solver adapter can consume.